# Step 1 ( Load Data ) : 


In [2]:
import pandas as pd
import numpy as np


df = pd.read_csv('smart_certified_sample.csv')



# Step 2 ( First Look ) :


In [3]:
print("=== Shape ===")
print(df.shape)

=== Shape ===
(2112743, 5)


In [4]:
print("\n=== First 5 Rows ===")
print(df.head())


=== First 5 Rows ===
   User_ID  Product_ID  Category_ID Behavior   Timestamp
0  1000283     4970571      4145813       pv  1511681761
1  1000283     3263745       982926     cart  1511705117
2  1000283     4795757       982926       pv  1511705505
3  1000283     3263745       982926      buy  1511756717
4  1000283     3578429       982926       pv  1511757021


In [5]:
print("\n=== Data Types ===")
print(df.dtypes)


=== Data Types ===
User_ID        int64
Product_ID     int64
Category_ID    int64
Behavior         str
Timestamp      int64
dtype: object


In [6]:
print("\n=== Missing Values ===")
print(df.isnull().sum())


=== Missing Values ===
User_ID        0
Product_ID     0
Category_ID    0
Behavior       0
Timestamp      0
dtype: int64


In [7]:
print("\n=== Behavior Distribution ===")
print(df['Behavior'].value_counts())
print("\nPercentages:")
print(df['Behavior'].value_counts(normalize=True) * 100)


=== Behavior Distribution ===
Behavior
pv      1892462
cart     117862
fav       60416
buy       42003
Name: count, dtype: int64

Percentages:
Behavior
pv      89.573696
cart     5.578625
fav      2.859600
buy      1.988079
Name: proportion, dtype: float64


In [8]:
print("\n=== Unique Values ===")
print(f"Unique Users     : {df['User_ID'].nunique():,}")
print(f"Unique Products  : {df['Product_ID'].nunique():,}")
print(f"Unique Categories: {df['Category_ID'].nunique():,}")


=== Unique Values ===
Unique Users     : 20,000
Unique Products  : 659,797
Unique Categories: 6,660


In [9]:
print("\n=== Basic Stats ===")
print(df.describe())


=== Basic Stats ===
            User_ID    Product_ID   Category_ID     Timestamp
count  2.112743e+06  2.112743e+06  2.112743e+06  2.112743e+06
mean   5.039641e+05  2.580720e+06  2.697946e+06  1.511961e+09
std    2.958048e+05  1.489033e+06  1.464081e+06  3.033607e+05
min    2.100000e+01  9.000000e+00  2.171000e+03  1.496323e+09
25%    2.447380e+05  1.293698e+06  1.320530e+06  1.511761e+09
50%    5.012030e+05  2.583963e+06  2.693696e+06  1.511965e+09
75%    7.572620e+05  3.865656e+06  4.145813e+06  1.512179e+09
max    1.017973e+06  5.163064e+06  5.161669e+06  1.634892e+09


# - Cleaning : 

### Step 1: Deleting cart


In [10]:
print("Before:", df.shape)
df = df[df['Behavior'] != 'cart']
print("After removing cart:", df.shape)

Before: (2112743, 5)
After removing cart: (1994881, 5)


### Step 2:  Convert  Timestamp

In [11]:
df['DateTime'] = pd.to_datetime(df['Timestamp'], unit='s')
df = df.drop(columns=['Timestamp'])
print("\nDate Range (before filter):")
print(f"From : {df['DateTime'].min()}")
print(f"To   : {df['DateTime'].max()}")


Date Range (before filter):
From : 2017-06-01 13:12:47
To   : 2021-10-22 08:32:06


### Step 3: Delete Wrong Dates


In [12]:
df = df[(df['DateTime'] >= '2017-11-25') & 
        (df['DateTime'] <= '2017-12-03')]
print("\nDate Range (after filter):")
print(f"From : {df['DateTime'].min()}")
print(f"To   : {df['DateTime'].max()}")


Date Range (after filter):
From : 2017-11-25 00:00:00
To   : 2017-12-03 00:00:00


### Step 4: Interaction Score


In [13]:
score_map = {'pv': 1, 'fav': 5, 'buy': 10}
df['Score'] = df['Behavior'].map(score_map)
print("\nScore Distribution:")
print(df['Score'].value_counts())


Score Distribution:
Score
1     1641300
5       52595
10      36700
Name: count, dtype: int64


### Step 5: Deleting Duplicates


In [14]:
print(f"\nDuplicates: {df.duplicated().sum():,}")
df = df.drop_duplicates()


Duplicates: 3


### Step 6: Final Check

In [15]:
print("\n=== Final Shape ===")
print(df.shape)
print(f"Users    : {df['User_ID'].nunique():,}")
print(f"Avg/User : {len(df)/df['User_ID'].nunique():.1f}")
print(f"Buy count: {(df['Behavior']=='buy').sum():,}")


=== Final Shape ===
(1730592, 6)
Users    : 19,995
Avg/User : 86.6
Buy count: 36,700


In [17]:
#df['DateTime'] = pd.to_datetime(df['Timestamp'], unit='s')
print(df['DateTime'].describe())
print("\nValue counts by month:")
print(df['DateTime'].dt.to_period('M').value_counts().sort_index())

count                1730592
mean     2017-11-29 01:54:17
min      2017-11-25 00:00:00
25%      2017-11-27 01:23:17
50%      2017-11-29 05:08:50
75%      2017-12-01 06:26:08
max      2017-12-03 00:00:00
Name: DateTime, dtype: object

Value counts by month:
DateTime
2017-11    1234077
2017-12     496515
Freq: M, Name: count, dtype: int64


In [18]:
df.to_csv('smart_cleaned.csv', index=False)
print("✅ Saved as smart_cleaned.csv!")
print(f"\nFinal Summary:")
print(f"Rows     : {len(df):,}")
print(f"Users    : {df['User_ID'].nunique():,}")
print(f"Avg/User : {len(df)/df['User_ID'].nunique():.1f}")
print(f"Buy count: {(df['Behavior']=='buy').sum():,}")

✅ Saved as smart_cleaned.csv!

Final Summary:
Rows     : 1,730,592
Users    : 19,995
Avg/User : 86.6
Buy count: 36,700
